In [3]:
# MLP Pytorch实现
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset


In [4]:
# 1.设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
# 2.数据加载与处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,),(0.3530,))
])

train_dataset = torchvision.datasets.FashionMNIST(root='../data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='../data', train=False, transform=transform, download=True)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [12]:
# 2.模型定义
class MLP(nn.Module):
    def __init__(self, input_size = 784, hidden_size = 256, output_size = 10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )
    
    def forward(self, X):
        X = self.flatten(X)
        return self.network(X)
model = MLP().to(device)


In [20]:
# 3.模型训练
def train_model(model, train_loader, test_loader, num_epochs=10, lr=0.1):
    # 1.定义损失函数和优化器
    loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # 2.记录训练过程
    train_losses, train_accs = [], []
    test_losses, test_accs = [], []

    # 3.训练模型
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct_sample, total_sample = 0.0, 0, 0
    
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            # 1.前向传播
            outputs = model(inputs)
            loss_value = loss(outputs, targets)

            # 2.反向传播和优化
            optimizer.zero_grad()
            loss_value.backward()
            optimizer.step()

            # 3.统计训练信息
            running_loss += loss_value.item()
            _, predicted = outputs.max(1)
            total_sample += targets.size(0)
            correct_sample += predicted.eq(targets).sum().item()

            # 4.打印进度
            if batch_idx % 100 == 0:
                print(f'Epoch: {epoch+1}/{num_epochs} | '
                      f'Batch: {batch_idx}/{len(train_loader)} | '
                      f'Loss: {loss_value.item():.4f}')
        # 计算训练集准确率
        train_acc = correct_sample / total_sample
        train_losses.append(running_loss / len(train_loader))
        train_accs.append(train_acc)

        # 测试阶段
        test_loss, test_acc = evaluate_model(model, test_loader, loss)
        test_losses.append(test_loss)
        test_accs.append(test_acc)

        # 打印测试集准确率
        print(f'Epoch: {epoch+1}/{num_epochs} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}')
    return train_losses, train_accs, test_losses, test_accs

# 4.模型评估
def evaluate_model(model, test_loader, loss):
    model.eval()
    running_loss, correct_sample, total_sample = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss_value = loss(outputs, targets)
            running_loss += loss_value.item()
            _, predicted = outputs.max(1)
            total_sample += targets.size(0)
            correct_sample += predicted.eq(targets).sum().item()
    test_loss = running_loss / len(test_loader)
    test_acc = correct_sample / total_sample
    return test_loss, test_acc

In [21]:
# 5.执行训练
print("Starting training...")
train_losses, train_accs, test_losses, test_accs = train_model(
    model, train_loader, test_loader, num_epochs=10, lr=0.1
)

print("\nTraining completed!")

Starting training...
Epoch: 1/10 | Batch: 0/235 | Loss: 2.0093
Epoch: 1/10 | Batch: 100/235 | Loss: 0.5734
Epoch: 1/10 | Batch: 200/235 | Loss: 0.4974
Epoch: 1/10 | Test Loss: 0.4739 | Test Acc: 0.8264
Epoch: 2/10 | Batch: 0/235 | Loss: 0.4929
Epoch: 2/10 | Batch: 100/235 | Loss: 0.4510
Epoch: 2/10 | Batch: 200/235 | Loss: 0.3666
Epoch: 2/10 | Test Loss: 0.5037 | Test Acc: 0.8115
Epoch: 3/10 | Batch: 0/235 | Loss: 0.5001
Epoch: 3/10 | Batch: 100/235 | Loss: 0.3775
Epoch: 3/10 | Batch: 200/235 | Loss: 0.4742
Epoch: 3/10 | Test Loss: 0.4706 | Test Acc: 0.8207
Epoch: 4/10 | Batch: 0/235 | Loss: 0.6030
Epoch: 4/10 | Batch: 100/235 | Loss: 0.3998
Epoch: 4/10 | Batch: 200/235 | Loss: 0.2968
Epoch: 4/10 | Test Loss: 0.4602 | Test Acc: 0.8300
Epoch: 5/10 | Batch: 0/235 | Loss: 0.4342
Epoch: 5/10 | Batch: 100/235 | Loss: 0.3851
Epoch: 5/10 | Batch: 200/235 | Loss: 0.2792
Epoch: 5/10 | Test Loss: 0.3892 | Test Acc: 0.8566
Epoch: 6/10 | Batch: 0/235 | Loss: 0.3255
Epoch: 6/10 | Batch: 100/235 | L